In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import hstack
from sklearn.metrics import ndcg_score
import xgboost as xgb
import joblib

ROOT = Path.cwd().parents[0]
DATA = ROOT / "models"
OUT  = ROOT / "models"
OUT.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA / "train_mentor.csv")
test  = pd.read_csv(DATA / "test_mentor.csv")

print(f"Dataset sizes: train={len(train)}, test={len(test)}")

Dataset sizes: train=288154, test=31286


In [2]:
def to_list(s: str):
    return [x.strip().lower() for x in str(s).split(";") if x.strip()]

mlb_int = MultiLabelBinarizer(sparse_output=True)
mlb_tag = MultiLabelBinarizer(sparse_output=True)

X_train = hstack([
    mlb_int.fit_transform(train["field_tags"].map(to_list)),
    mlb_tag.fit_transform(train["expertise_tags"].map(to_list))
], format="csr")
y_train = train["label_match"].values

X_test = hstack([
    mlb_int.transform(test["field_tags"].map(to_list)),
    mlb_tag.transform(test["expertise_tags"].map(to_list))
], format="csr")
y_test  = test["label_match"].values

print("XGBoost feature shapes:", X_train.shape, X_test.shape)

XGBoost feature shapes: (288154, 48) (31286, 48)


In [3]:
params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "max_depth": 10,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "nthread": -1,
    "seed": 42
}

In [4]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test, label=y_test)

print("Training XGBoostRegressor ...")
evals = [(dtrain, "train"), (dtest, "test")]
model = xgb.train(params, dtrain, num_boost_round=300, evals=evals, verbose_eval=50)

test_pred = model.predict(dtest)


Training XGBoostRegressor ...
[0]	train-rmse:0.09251	test-rmse:0.09062
[50]	train-rmse:0.03782	test-rmse:0.03878
[100]	train-rmse:0.02466	test-rmse:0.02656
[150]	train-rmse:0.01981	test-rmse:0.02267
[200]	train-rmse:0.01743	test-rmse:0.02121
[250]	train-rmse:0.01583	test-rmse:0.02047
[299]	train-rmse:0.01467	test-rmse:0.01992


In [5]:

model.save_model(OUT / "xgb_mentor_labelmatch_regressor.json")
print(f"Model saved: {OUT / 'xgb_mentor_labelmatch_regressor.json'}")

bundle = {
    "model": model,
    "mlb_int": mlb_int,
    "mlb_tag": mlb_tag,
}
joblib.dump(bundle, OUT / "xgb_mentor_labelmatch_regressor.pkl")
print(f"Bundle saved: {OUT / 'xgb_mentor_labelmatch_regressor.pkl'}")


Model saved: /Users/zyh/Desktop/HDG_group/models/xgb_mentor_labelmatch_regressor.json
Bundle saved: /Users/zyh/Desktop/HDG_group/models/xgb_mentor_labelmatch_regressor.pkl


In [6]:
out_df = test.copy()
out_df["pred_label_match"] = np.round(test_pred, 4)
out_df.to_csv(OUT / "xgb_mentor_test.csv", index=False)

In [7]:

val_df = test[["program_id", "label_match"]].copy()
val_df["pred_label_match"] = np.asarray(test_pred, dtype=float)

rows = []
for sid, g in val_df.groupby("program_id", sort=True):
    y_true = g["label_match"].to_numpy().reshape(1, -1)
    y_pred = g["pred_label_match"].to_numpy().reshape(1, -1)
    rows.append({"program_id": sid, "nDCG@3": float(ndcg_score(y_true, y_pred, k=3))})

ndcg_XGB = pd.DataFrame(rows).sort_values("program_id").reset_index(drop=True)
mean_ndcg_XGB = float(ndcg_XGB["nDCG@3"].mean())

display(ndcg_XGB)
print("Mean nDCG@3", round(mean_ndcg_XGB, 6))

,program_id,nDCG@3
0,8,0.968436
1,44,1.000000
2,45,1.000000
3,57,0.959184
4,58,0.976378
...,...,...
125,1309,0.950880
126,1319,0.991027
127,1328,1.000000
128,1329,0.988805


Mean nDCG@3 0.978267
